In [1]:
import pandas as pd

# Check karo file exists hai ya nahi
import os
if os.path.exists('gpx4_inhibitors.csv'):
    df = pd.read_csv('gpx4_inhibitors.csv')
    print(f"✅ File mil gayi! {len(df)} records hain")
    print(df.head())
else:
    print("❌ File nahi mili - dobara download karna padega")

✅ File mil gayi! 11 records hain
                                    canonical_smiles molecule_chembl_id  \
0       COc1cc(N(C(=O)CCl)c2ccc3c(c2)OCCO3)cc(OC)c1F      CHEMBL5422297   
1       COc1cc(N(C(=O)CBr)c2ccc3c(c2)OCCO3)cc(OC)c1F      CHEMBL5415644   
2  COc1ccc2c(c1)NC(=O)/C2=C1\Nc2ccccc2\C1=N/OCCCN...      CHEMBL5570241   
3  COc1ccc(N(C(=O)CCl)C(C(=O)NCCc2ccccc2)c2cccs2)...      CHEMBL1499544   
4  COc1ccc(N(C(=O)CCl)C(C(=O)NCCc2ccccc2)c2cccs2)...      CHEMBL1499544   

  standard_units  standard_value units   value  IC50_nM  
0             nM           120.0    uM    0.12    120.0  
1             nM           130.0    uM    0.13    130.0  
2             nM           542.5    nM  542.50    542.5  
3             nM          2700.0    uM    2.70   2700.0  
4             nM          4850.0    uM    4.85   4850.0  


In [2]:
from chembl_webresource_client.new_client import new_client
import pandas as pd

# Molecule database
molecule = new_client.molecule

# Selenium-containing molecules dhoondho
# SMARTS pattern use karenge
selenium_compounds = molecule.filter(
    molecule_structures__canonical_smiles__contains='[Se]'
).only(['molecule_chembl_id', 'pref_name', 
        'molecule_structures', 'max_phase'])

# List banao - sirf top 20
results = list(selenium_compounds[:20])
print(f"✅ Found {len(results)} selenium compounds (showing top 20)")

✅ Found 20 selenium compounds (showing top 20)


In [3]:
import pandas as pd

# Data extract karo
selenium_data = []

for m in results:
    if m['molecule_structures']:
        smiles = m['molecule_structures'].get('canonical_smiles', '')
        if '[Se]' in smiles or 'Se' in smiles:
            selenium_data.append({
                'ChEMBL_ID': m['molecule_chembl_id'],
                'Name': m['pref_name'] if m['pref_name'] else 'Unnamed',
                'SMILES': smiles,
                'Max_Phase': m['max_phase'] if m['max_phase'] else 0
            })

# DataFrame banao
se_df = pd.DataFrame(selenium_data)

print(f"✅ Selenium library created: {len(se_df)} compounds")
print()
se_df

✅ Selenium library created: 20 compounds



,ChEMBL_ID,Name,SMILES,Max_Phase
0,CHEMBL13581,Unnamed,CC(C)(C)C1=CC(=C/C=C/c2cc(C(C)(C)C)[se+]c(C(C)...,0
1,CHEMBL13434,Unnamed,CC(C)(C)C1=CC(=C/C=C/c2cc(C(C)(C)C)[te+]c(C(C)...,0
2,CHEMBL14104,Unnamed,C1CCN(CC[Se]CCN2CCCCC2)CC1,0
3,CHEMBL14259,Unnamed,C[N+](C)(C)CC[Se]CC[N+](C)(C)C,0
4,CHEMBL14142,Unnamed,CN(C)CC[Se]CCN(C)C,0
5,CHEMBL14131,Unnamed,CC(C)N(CC[Se]CCN(C(C)C)C(C)C)C(C)C,0
6,CHEMBL14118,Unnamed,C1CN(CC[Se]CCN2CCOCC2)CCO1,0
7,CHEMBL20224,Unnamed,CC1(C)N=C(N)N=C(N)N1c1cccc(C[Se]c2ccccc2)c1,0
8,CHEMBL25117,Unnamed,C[C@]12CC[C@H]3[C@@H](CCC4=CC(=O)CC[C@@]43C)[C...,0
9,CHEMBL416383,Unnamed,CC(=O)[C@@]1([Se]c2ccccc2)CC[C@H]2[C@@H]3CCC4=...,0


In [4]:
from rdkit import Chem
from rdkit.Chem import DataStructs
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

mfpgen = GetMorganGenerator(radius=2)

# UDS1b
uds1b_mol = Chem.MolFromSmiles('C1=CC=CC=C1C(C[Se]C(C2=CC=CC=C2)=O)=O')
uds1b_fp = mfpgen.GetFingerprint(uds1b_mol)

# Sab selenium compounds se compare karo
similarities = []
for smiles in se_df['SMILES']:
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        fp = mfpgen.GetFingerprint(mol)
        sim = DataStructs.TanimotoSimilarity(uds1b_fp, fp)
        similarities.append(round(sim * 100, 2))
    else:
        similarities.append(0)

se_df['Similarity_to_UDS1b_%'] = similarities

# Sort
se_df_sorted = se_df.sort_values('Similarity_to_UDS1b_%', ascending=False).reset_index(drop=True)

print("🔬 Selenium compounds ranked by similarity to UDS1b:\n")
se_df_sorted[['ChEMBL_ID', 'Name', 'Similarity_to_UDS1b_%']]

🔬 Selenium compounds ranked by similarity to UDS1b:



,ChEMBL_ID,Name,Similarity_to_UDS1b_%
0,CHEMBL39469,BENZENE SELENOIC ACID,25.81
1,CHEMBL446091,Unnamed,23.53
2,CHEMBL38761,Unnamed,22.22
3,CHEMBL25117,Unnamed,18.46
4,CHEMBL39437,Unnamed,17.78
5,CHEMBL416383,Unnamed,15.62
6,CHEMBL20224,Unnamed,13.79
7,CHEMBL37964,Unnamed,13.48
8,CHEMBL285757,Unnamed,12.37
9,CHEMBL37569,Unnamed,11.63
